# Bölüm 7 — BÖLÜM 7: GÖZETİMSİZ ÖĞRENME: KÜMELEME VE BOYUT İNDİRGEME

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 7. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q hdbscan matplotlib numpy pandas scikit-learn scipy umap-learn


## 7.1. Kümeleme Analizi (Cluster Analysis)


### Python Uygulaması — Kapsamlı K-Means Analizi

`bolum07/07_01_01_python-uygulamasi-kapsamli-k-means-analizi.py`

_Kitap: Kod 7.1_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# --- Python: Veri Hazırlama ve Temel K-Means ---
# ─── 1. Sentetik Veri Seti (4 doğal küme) ────────────────────────
X, y_true = make_blobs(
    n_samples=600, centers=4, cluster_std=0.85, random_state=42)

# --- Python: Veri Hazırlama ve Temel K-Means ---
# Kümeleme öncesi MUTLAKA standartlaştır
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Python: Veri Hazırlama ve Temel K-Means ---
# ─── 2. K Seçimi: Elbow + Silhouette + DBI ────────────────────────
wcss, sil_scores, dbi_scores = [], [], []
K_range = range(2, 12)

# --- Python: Veri Hazırlama ve Temel K-Means ---
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    dbi_scores.append(davies_bouldin_score(X_scaled, labels))

# --- Python: Veri Hazırlama ve Temel K-Means ---
print("K | WCSS      | Silhouette | DBI")
for i, k in enumerate(K_range):
    print(f"{k:2d}| {wcss[i]:9.2f}| {sil_scores[i]:10.4f}| {dbi_scores[i]:.4f}")

# --- Python: Veri Hazırlama ve Temel K-Means ---
# Grafik
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(K_range, wcss, "o--", color="steelblue")
axes[0].set_title("Elbow Yöntemi (WCSS)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("WCSS")

# --- Python: Veri Hazırlama ve Temel K-Means ---
axes[1].plot(K_range, sil_scores, "s--", color="forestgreen")
axes[1].set_title("Silhouette Skoru (↑ iyi)")
axes[1].set_xlabel("K")

# --- Python: Veri Hazırlama ve Temel K-Means ---
axes[2].plot(K_range, dbi_scores, "^--", color="crimson")
axes[2].set_title("Davies-Bouldin İndeksi (↓ iyi)")
axes[2].set_xlabel("K")

# --- Python: Veri Hazırlama ve Temel K-Means ---
plt.tight_layout()
plt.show()

import time
from sklearn.metrics import silhouette_samples

# ─── 3. En İyi K ile Nihai Model ─────────────────────────────────
best_k = 4   # Metriklerden belirlendi
final_km = KMeans(n_clusters=best_k, init="k-means++",
                  n_init=15, max_iter=300, random_state=42)
labels = final_km.fit_predict(X_scaled)

print(f"Nihai Model — K={best_k}")
print(f"  WCSS (Eylemsizlik): {final_km.inertia_:.2f}")
print(f"  Silhouette Skoru:   {silhouette_score(X_scaled, labels):.4f}")
print(f"  Davies-Bouldin:     {davies_bouldin_score(X_scaled, labels):.4f}")
print(f"  İterasyon Sayısı:   {final_km.n_iter_}")

# Küme boyutları
unique, counts = np.unique(labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  Küme {c}: {n} örnek")

# ─── 4. Silhouette Görselleştirmesi ───────────────────────────────
sil_vals = silhouette_samples(X_scaled, labels)
avg_sil  = silhouette_score(X_scaled, labels)

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
cmap = plt.get_cmap("tab10")   # plt.cm.get_cmap matplotlib 3.11'de kaldirildi

for i in range(best_k):
    sil_i = np.sort(sil_vals[labels == i])
    size_i = sil_i.shape[0]
    y_upper = y_lower + size_i
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                     facecolor=cmap(i), alpha=0.75)
    ax.text(-0.05, y_lower + 0.5 * size_i, f"Küme {i}")
    y_lower = y_upper + 10

ax.axvline(x=avg_sil, color="red", linestyle="--")
ax.set_title(f"Silhouette Grafiği (K={best_k}, Ort={avg_sil:.3f})")
ax.set_xlabel("Silhouette Katsayısı")
ax.set_ylabel("Küme Etiketi")
plt.tight_layout(); plt.show()

# ─── 5. Mini-Batch K-Means (Büyük Veri) ──────────────────────────
t0 = time.time()
km_full = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_scaled)
print(f"KMeans süresi:       {time.time()-t0:.3f}s")

t0 = time.time()
mb_km = MiniBatchKMeans(n_clusters=best_k, batch_size=128,
                         n_init=10, random_state=42).fit(X_scaled)
print(f"MiniBatch KMeans:    {time.time()-t0:.3f}s")
print(f"MiniBatch Sil Skoru: {silhouette_score(X_scaled, mb_km.labels_):.4f}")


### Python Uygulaması — Hiyerarşik Kümeleme

`bolum07/07_01_02_python-uygulamasi-hiyerarsik-kumeleme.py`

_Kitap: Kod 7.2_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np

# ─── 1. Veri ──────────────────────────────────────────────────────
X, _ = make_blobs(n_samples=200, centers=4,
                   cluster_std=0.6, random_state=42)
X_sc = StandardScaler().fit_transform(X)

# ─── 2. Dendrogram Çizimi (Ward bağlantısı) ──────────────────────
linkage_matrix = sch.linkage(X_sc, method="ward")

plt.figure(figsize=(14, 6))
sch.dendrogram(
    linkage_matrix,
    truncate_mode="lastp",   # Yalnızca son p birleşimi göster
    p=30,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True
)
plt.title("Agglomerative Kümeleme Dendrogramı (Ward Bağlantısı)")
plt.xlabel("Veri Noktaları (veya Alt Küme Boyutu)")
plt.ylabel("Birleşme Mesafesi (Ward)")
# Kesim seviyesini dendrogram üzerine işaretle
plt.axhline(y=6.5, color="red", linestyle="--", linewidth=2, label="K=4 kesim")
plt.legend(); plt.tight_layout(); plt.show()

# ─── 3. Dört Bağlantı Türünü Karşılaştır ─────────────────────────
linkages = ["ward", "complete", "average", "single"]

# Bagalanti olcutlerinin FARKINI gorebilmek icin anizotropik ve degisken
# yogunluklu veri gerekir. Kusursal, esit yayilimli kumelerde dort olcut de
# ayni sonucu verir ve karsilastirma ogretici olmaz.
X_anz, _ = make_blobs(n_samples=200, centers=4,
                       cluster_std=[1.0, 2.5, 0.5, 1.5], random_state=42)
X_anz = X_anz @ np.array([[0.6, -0.6], [-0.4, 0.85]])   # egme donusumu
X_anz_sc = StandardScaler().fit_transform(X_anz)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, linkage_type in zip(axes, linkages):
    agg = AgglomerativeClustering(
        n_clusters=4, metric="euclidean", linkage=linkage_type)
    labels = agg.fit_predict(X_anz_sc)
    sil = silhouette_score(X_anz_sc, labels)
    ax.scatter(X_anz_sc[:, 0], X_anz_sc[:, 1], c=labels, cmap="tab10", s=30)
    ax.set_title(f"{linkage_type.capitalize()} Linkage\nSil={sil:.3f}")
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

# ─── 4. Farklı K Değerleri için Silhouette Karşılaştırması ───────
K_vals = range(2, 10)
sil_ward = []

for k in K_vals:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    lbl = agg.fit_predict(X_sc)
    sil_ward.append(silhouette_score(X_sc, lbl))

best_k = K_vals[np.argmax(sil_ward)]
print(f"Ward + Silhouette ile optimal K: {best_k}")

# ─── 5. Mesafe Matrisine Dayalı Kümeleme (cosine metriği) ─────────
# Ward yalnızca Euclidean destekler; farklı metrik için average/complete kullan
agg_cos = AgglomerativeClustering(
    n_clusters=4, metric="cosine", linkage="average")
labels_cos = agg_cos.fit_predict(X_sc)
print(f"Cosine metric Silhouette: {silhouette_score(X_sc, labels_cos):.4f}")


### DBSCAN Algoritması (Pseudocode)

`bolum07/07_01_03_dbscan-algoritmasi.py`


In [ ]:
def DBSCAN(D, eps, MinPts):
    labels = {p: UNDEFINED for p in D}
    cluster_id = 0

# --- Algoritma: DBSCAN Pseudocode ---
    for p in D:
        if labels[p] != UNDEFINED: continue  # Zaten işlendi

# --- Algoritma: DBSCAN Pseudocode ---
        neighbors = range_query(D, p, eps)    # ε-komşuluğu bul

# --- Algoritma: DBSCAN Pseudocode ---
        if len(neighbors) < MinPts:           # Çekirdek değil
            labels[p] = NOISE                 # Geçici gürültü
            continue

# --- Algoritma: DBSCAN Pseudocode ---
        # Yeni küme başlat
        cluster_id += 1
        labels[p] = cluster_id

# --- Algoritma: DBSCAN Pseudocode ---
        # Tüm ulaşılabilir noktaları kümeye ekle (BFS/DFS)
        seed_set = set(neighbors) - {p}
        while seed_set:
            q = seed_set.pop()
            if labels[q] == NOISE: labels[q] = cluster_id
            if labels[q] != UNDEFINED: continue
            labels[q] = cluster_id
            q_neighbors = range_query(D, q, eps)
            if len(q_neighbors) >= MinPts:
                seed_set |= set(q_neighbors)  # Çekirdek — genişlet

# --- Algoritma: DBSCAN Pseudocode ---
    return labels


### Python Uygulaması — DBSCAN ve HDBSCAN

`bolum07/07_01_03_python-uygulamasi-dbscan-ve-hdbscan.py`

_Kitap: Kod 7.3_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# ─── 1. Şekil Bağımsız Kümeleme Testi ────────────────────────────
datasets = {
    "Hilal (Moons)"  : make_moons(n_samples=300, noise=0.05, random_state=42),
    "Halkalar (Circles)": make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42),
    "Kümeler"        : make_blobs(n_samples=300, centers=4, cluster_std=0.5, random_state=42),
}

params = {
    "Hilal (Moons)"     : {"eps": 0.3,  "min_samples": 5},
    "Halkalar (Circles)": {"eps": 0.35, "min_samples": 5},   # 0.2 cok kucuktu: 18 parca
    "Kümeler"           : {"eps": 0.5,  "min_samples": 5},
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (name, (X_d, _)) in enumerate(datasets.items()):
    X_sc_d = StandardScaler().fit_transform(X_d)

    # K-Means
    from sklearn.cluster import KMeans
    km_lbl = KMeans(n_clusters=2, random_state=42).fit_predict(X_sc_d)
    axes[0, col].scatter(X_sc_d[:,0], X_sc_d[:,1], c=km_lbl, cmap="tab10", s=20)
    axes[0, col].set_title(f"K-Means: {name}")

    # DBSCAN
    p = params[name]
    db_lbl = DBSCAN(**p).fit_predict(X_sc_d)
    n_clusters = len(set(db_lbl)) - (1 if -1 in db_lbl else 0)
    n_noise = (db_lbl == -1).sum()
    axes[1, col].scatter(X_sc_d[:,0], X_sc_d[:,1], c=db_lbl, cmap="tab10", s=20)
    axes[1, col].set_title(f"DBSCAN: {name}\n(K={n_clusters}, Gürültü={n_noise})")

plt.tight_layout(); plt.show()

# ─── 2. Parametre Izgarası ile DBSCAN Optimizasyonu ───────────────
X_bl, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.7, random_state=42)
X_bl_sc = StandardScaler().fit_transform(X_bl)

results = []
for eps in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]:
    for min_pts in [3, 5, 7, 10]:
        lbl = DBSCAN(eps=eps, min_samples=min_pts).fit_predict(X_bl_sc)
        n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
        n_n = (lbl == -1).sum()
        if n_c > 1:   # Silhouette gürültüsüz noktalarda hesaplanır
            mask = lbl != -1
            sil = silhouette_score(X_bl_sc[mask], lbl[mask]) if mask.sum() > 10 else 0
        else:
            sil = -1
        results.append({"eps": eps, "min_pts": min_pts,
                         "n_clusters": n_c, "n_noise": n_n, "sil": sil})

import pandas as pd
df_res = pd.DataFrame(results)
best = df_res.loc[df_res.sil.idxmax()]
print(f"En iyi params → eps={best.eps}, min_pts={best.min_pts:.0f}")
print(f"Kümeler: {best.n_clusters:.0f}, Gürültü: {best.n_noise:.0f}, Sil: {best.sil:.4f}")

# --- Python: HDBSCAN Uygulaması ---
# pip install hdbscan
import hdbscan

# --- Python: HDBSCAN Uygulaması ---
X_m, _ = make_moons(n_samples=400, noise=0.08, random_state=42)
X_m_sc = StandardScaler().fit_transform(X_m)

# --- Python: HDBSCAN Uygulaması ---
# HDBSCAN — ε yoktur; min_cluster_size yeterli
hdb = hdbscan.HDBSCAN(
    min_cluster_size=15,   # Minimum küme boyutu
    min_samples=5,         # Çekirdek nokta için min komşu
    cluster_selection_epsilon=0.0,  # 0 = tam hiyerarşik seçim
    prediction_data=True)

# --- Python: HDBSCAN Uygulaması ---
hdb_labels = hdb.fit_predict(X_m_sc)

# --- Python: HDBSCAN Uygulaması ---
n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print(f"HDBSCAN Kümeler: {n_clusters}, Gürültü Noktaları: {n_noise}")

# --- Python: HDBSCAN Uygulaması ---
# Yumuşak kümeleme — olasılık skoru
soft_clusters = hdbscan.all_points_membership_vectors(hdb)
print(f"İlk 5 noktanın küme üyelik olasılıkları:\n{soft_clusters[:5].round(3)}")

# --- Python: HDBSCAN Uygulaması ---
# Küme kararlılık skoru
print(f"Küme Kararlılıkları: {hdb.cluster_persistence_}")

# --- Python: HDBSCAN Uygulaması ---
# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Python: HDBSCAN Uygulaması ---
sc1 = axes[0].scatter(X_m_sc[:,0], X_m_sc[:,1],
                      c=hdb_labels, cmap="tab10", s=30)
axes[0].set_title(f"HDBSCAN Etiketleri (K={n_clusters})")

# --- Python: HDBSCAN Uygulaması ---
# Renk = üyelik olasılığı
axes[1].scatter(X_m_sc[:,0], X_m_sc[:,1],
               c=hdb.probabilities_, cmap="viridis", s=30)
plt.colorbar(axes[1].scatter(X_m_sc[:,0], X_m_sc[:,1],
             c=hdb.probabilities_, cmap="viridis", s=30),
             ax=axes[1], label="Küme Olasılığı")
axes[1].set_title("HDBSCAN Yumuşak Kümeleme Olasılıkları")

# --- Python: HDBSCAN Uygulaması ---
plt.tight_layout(); plt.show()


### ε Seçimi: k-NN Mesafe Grafiği

`bolum07/07_01_03_secimi-k-nn-mesafe-grafigi.py`


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum07/07_01_03_python-uygulamasi-dbscan-ve-hdbscan.py
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
X_ham, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
X_sc = StandardScaler().fit_transform(X_ham)
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
# --- Python: k-NN Mesafe Grafiği ile ε Seçimi ---
from sklearn.neighbors import NearestNeighbors

# --- Python: k-NN Mesafe Grafiği ile ε Seçimi ---
MinPts = 5
k = MinPts - 1   # k. en yakın komşu mesafesi

# --- Python: k-NN Mesafe Grafiği ile ε Seçimi ---
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_sc)
distances, _ = nn.kneighbors(X_sc)

# --- Python: k-NN Mesafe Grafiği ile ε Seçimi ---
# k. komşu mesafeleri büyükten küçüğe sırala
k_distances = np.sort(distances[:, -1])[::-1]

# --- Python: k-NN Mesafe Grafiği ile ε Seçimi ---
plt.figure(figsize=(8, 4))
plt.plot(k_distances, linewidth=2)
plt.xlabel("Noktalar (büyükten küçüğe sıralı)")
plt.ylabel(f"{k}. En Yakın Komşu Mesafesi")
plt.title(f"k-NN Mesafe Grafiği (k={k}) — Dirsek Noktası = optimal ε")
# Dirsek noktasini elle secmek yerine egrinin en buyuk egrilik noktasindan turet.
# (Sabit 0.5 degeri egrinin cok uzerinde kaliyor ve grafigi yaniltici hale getiriyordu.)
# Dirsek: egrinin ilk ve son noktasini birlestiren kirise en uzak nokta
_x = np.arange(len(k_distances), dtype=float)
_y = k_distances.astype(float)
_xn = (_x - _x.min()) / (_x.max() - _x.min())
_yn = (_y - _y.min()) / (_y.max() - _y.min())
_p1 = np.array([_xn[0], _yn[0]])
_p2 = np.array([_xn[-1], _yn[-1]])
_vek = _p2 - _p1
_vek = _vek / np.linalg.norm(_vek)
_noktalar = np.column_stack([_xn, _yn]) - _p1
_izdusum = np.outer(_noktalar @ _vek, _vek)
_uzaklik = np.linalg.norm(_noktalar - _izdusum, axis=1)
_dirsek = int(np.argmax(_uzaklik))
eps_tahmin = float(_y[_dirsek])
plt.axhline(y=eps_tahmin, color="red", linestyle="--",
            label=f"ε ≈ {eps_tahmin:.3f} (dirsek noktası)")
plt.axvline(x=_dirsek, color="gray", linestyle=":", linewidth=1)
plt.legend(); plt.tight_layout(); plt.show()


## 7.2. Boyut İndirgeme (Dimensionality Reduction)


### Python Uygulaması — Kapsamlı PCA

`bolum07/07_02_01_python-uygulamasi-kapsamli-pca.py`

_Kitap: Kod 7.4_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
# ─── 1. Veri Hazırlama ────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)         # Standardizasyon zorunlu!

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
# ─── 2. Scree Plot — Bileşen Sayısı Seçimi ──────────────────────
pca_full = PCA(n_components=None)       # Tüm bileşenler
pca_full.fit(X_sc)

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
# Scree Plot (özdeğer grafiği)
axes[0].bar(range(1, len(evr)+1), evr, alpha=0.7, color="steelblue")
axes[0].step(range(1, len(evr)+1), cum_evr, color="crimson", linewidth=2)
axes[0].axhline(y=0.95, color="green", linestyle="--", label="%95 Varyans")
axes[0].set_title("Scree Plot — Açıklanan Varyans")
axes[0].set_xlabel("Bileşen Sayısı")
axes[0].set_ylabel("Varyans Oranı")
axes[0].legend()

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
# Kümülatif varyans
k_95 = np.argmax(cum_evr >= 0.95) + 1
k_80 = np.argmax(cum_evr >= 0.80) + 1
print(f"%80 varyans için gereken bileşen: {k_80} / {X.shape[1]}")
print(f"%95 varyans için gereken bileşen: {k_95} / {X.shape[1]}")

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
# Yükleme haritası — PC1 ve PC2 hangi özellikleri taşıyor?
pca_k = PCA(n_components=k_80)
X_pca = pca_k.fit_transform(X_sc)

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
loadings_df = pd.DataFrame(
    pca_k.components_[:2].T,
    index=feature_names,
    columns=["PC1", "PC2"]
)
axes[1].imshow(loadings_df.values, cmap="RdBu_r", aspect="auto")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["PC1", "PC2"])
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels(feature_names, fontsize=7)
axes[1].set_title("PCA Yükleme Haritası (PC1–PC2)")

# --- Python: PCA — Scree Plot, Bileşen Seçimi ve Yorumlama ---
plt.tight_layout(); plt.show()

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# ─── 3. PCA + Sınıflandırıcı Pipeline ───────────────────────────
# PCA'nın model performansına katkısını ölç
from sklearn.model_selection import cross_val_score

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
results = {}
for n_comp in [2, 5, 10, k_80, X.shape[1]]:
    if n_comp == X.shape[1]:
        pipe = Pipeline([("scaler", StandardScaler()),
                          ("clf",    LogisticRegression(max_iter=1000))])
        label = f"Tüm boyutlar ({n_comp})"
    else:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=n_comp)),
            ("clf",    LogisticRegression(max_iter=1000))
        ])
        label = f"PCA({n_comp})"
    score = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean()
    results[label] = score
    print(f"{label:25s}: ROC-AUC = {score:.4f}")

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# ─── 4. Kernel PCA — Doğrusal Olmayan Projeksiyon ────────────────
from sklearn.decomposition import KernelPCA

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# RBF kernel ile doğrusal olmayan projeksiyon
kpca = KernelPCA(
    n_components=2,
    kernel="rbf",
    gamma=0.05,
    fit_inverse_transform=True)   # Yeniden yapılandırma için

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
X_kpca = kpca.fit_transform(X_sc)

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# Orijinal (ilk 2 özellik)
axes[0].scatter(X_sc[:, 0], X_sc[:, 1], c=y, cmap="Set1", alpha=0.6, s=20)
axes[0].set_title("Orijinal (ilk 2 özellik)")

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# Standart PCA
X_std_pca = PCA(n_components=2).fit_transform(X_sc)
axes[1].scatter(X_std_pca[:, 0], X_std_pca[:, 1], c=y, cmap="Set1", alpha=0.6, s=20)
axes[1].set_title("Standart PCA (2 bileşen)")

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# Kernel PCA
axes[2].scatter(X_kpca[:, 0], X_kpca[:, 1], c=y, cmap="Set1", alpha=0.6, s=20)
axes[2].set_title("Kernel PCA (RBF kernel)")

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
for ax in axes:
    ax.set_xlabel("Boyut 1")
    ax.set_ylabel("Boyut 2")
plt.tight_layout(); plt.show()

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# ─── 5. PCA ile Gürültü Azaltma ──────────────────────────────────
digits = load_digits()
X_dig = StandardScaler().fit_transform(digits.data)

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
# %95 varyansı koruyarak boyut indir ve geri yansıt
pca_dn = PCA(n_components=0.95, svd_solver="full")
X_dn = pca_dn.fit_transform(X_dig)
X_reconstructed = pca_dn.inverse_transform(X_dn)

# --- Python: PCA ile Lojistik Regresyon Pipeline ve Kernel PCA ---
print(f"Orijinal boyut: {X_dig.shape[1]}")
print(f"Sıkıştırılmış boyut: {X_dn.shape[1]} (%95 varyans korunuyor)")
print(f"Sıkıştırma oranı: {X_dn.shape[1]/X_dig.shape[1]:.2%}")


### Python Uygulaması — t-SNE ve UMAP

`bolum07/07_02_02_python-uygulamasi-t-sne-ve-umap.py`

_Kitap: Kod 7.5, Kod 7.6_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ─── 1. Veri: Digits (8x8 piksel el yazısı rakamları) ─────────────
digits = load_digits()
X_dig = digits.data        # 1797 örnek, 64 özellik (piksel)
y_dig = digits.target      # 0–9 etiketleri

# Standartlaştır
X_dig_sc = StandardScaler().fit_transform(X_dig)

# ─── 2. t-SNE öncesinde PCA ile ön boyut indirgeme (hız için) ─────
# t-SNE için önerilen: önce PCA ile 50 bileşene indir, sonra t-SNE uygula
pca_pre = PCA(n_components=30, random_state=42)
X_pca_pre = pca_pre.fit_transform(X_dig_sc)
print(f"PCA ön işleme: 64 → 30 boyut, {pca_pre.explained_variance_ratio_.sum():.1%} varyans korundu")

# ─── 3. Farklı Perplexity Değerlerini Karşılaştır ─────────────────
perplexities = [5, 15, 30, 50]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(
        n_components=2,
        perplexity=perp,
        learning_rate="auto",
        init="pca",            # PCA başlatma daha kararlı
        max_iter=1000,
        random_state=42)
    X_tsne = tsne.fit_transform(X_pca_pre)

    sc = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                    c=y_dig, cmap="tab10", s=8, alpha=0.7)
    ax.set_title(f"t-SNE perplexity={perp}")
    ax.axis("off")

plt.colorbar(sc, ax=axes[-1], label="Rakam")
plt.suptitle("Digits Veri Seti: t-SNE Perplexity Karşılaştırması", y=1.02)
plt.tight_layout(); plt.show()

# ─── 4. PCA vs t-SNE Karşılaştırması ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PCA
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_dig_sc)
ax1.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y_dig, cmap="tab10", s=10, alpha=0.7)
ax1.set_title("PCA (2 bileşen)")
ax1.axis("off")

# t-SNE (optimal ayarlar)
tsne_best = TSNE(n_components=2, perplexity=30, init="pca",
                  learning_rate="auto", max_iter=1500, random_state=42)
X_tsne_best = tsne_best.fit_transform(X_pca_pre)
ax2.scatter(X_tsne_best[:, 0], X_tsne_best[:, 1],
            c=y_dig, cmap="tab10", s=10, alpha=0.7)
ax2.set_title("t-SNE (perplexity=30, max_iter=1500)")
ax2.axis("off")

plt.suptitle("PCA vs t-SNE: Digits Görselleştirme")
plt.tight_layout(); plt.show()

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# pip install umap-learn
import umap
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_digits

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# ─── 1. Temel UMAP Uygulaması ─────────────────────────────────────
digits = load_digits()
X_dig_sc = StandardScaler().fit_transform(digits.data)

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
umap_viz = umap.UMAP(
    n_components=2,
    n_neighbors=15,        # Lokal komşuluk boyutu
    min_dist=0.1,          # Minimum küme sıkışıklığı
    metric="euclidean",
    random_state=42)

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
X_umap = umap_viz.fit_transform(X_dig_sc)

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
plt.figure(figsize=(8, 6))
sc = plt.scatter(X_umap[:, 0], X_umap[:, 1],
                 c=digits.target, cmap="tab10", s=10, alpha=0.8)
plt.colorbar(sc, label="Rakam")
plt.title("UMAP: Digits Veri Seti (n_neighbors=15, min_dist=0.1)")
plt.axis("off"); plt.tight_layout(); plt.show()

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# ─── 2. n_neighbors ve min_dist Etkisi ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
params = [(5, 0.0), (15, 0.1), (50, 0.5),
           (15, 0.0), (15, 0.5), (15, 0.99)]

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
for ax, (nn, md) in zip(axes.ravel(), params):
    um = umap.UMAP(n_neighbors=nn, min_dist=md,
                    n_components=2, random_state=42)
    X_um = um.fit_transform(X_dig_sc)
    ax.scatter(X_um[:,0], X_um[:,1], c=digits.target,
               cmap="tab10", s=5, alpha=0.7)
    ax.set_title(f"n_neighbors={nn}, min_dist={md}")
    ax.axis("off")

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
plt.suptitle("UMAP Hiperparametre Etkisi")
plt.tight_layout(); plt.show()

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# ─── 3. UMAP Projeksiyon — Yeni Veri Noktası Dönüştürme ──────────
# UMAP'ın t-SNE'ye büyük üstünlüğü: transform() methodu
from sklearn.model_selection import train_test_split

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig_sc, digits.target, test_size=0.2, random_state=42)

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
umap_pipe = umap.UMAP(n_components=10, n_neighbors=15,
                       random_state=42)
X_tr_umap = umap_pipe.fit_transform(X_tr)   # Sadece train üzerinde fit!
X_te_umap = umap_pipe.transform(X_te)        # Test verisini dönüştür

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
print(f"UMAP projeksiyon: {X_tr.shape[1]} → {X_tr_umap.shape[1]} boyut")

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# ─── 4. UMAP + Sınıflandırıcı Pipeline ──────────────────────────
# UMAP sklearn API uyumlu — Pipeline içinde kullanılabilir
pipe_umap = Pipeline([
    ("scaler", StandardScaler()),
    ("umap",   umap.UMAP(n_components=10, n_neighbors=15, random_state=42)),
    ("clf",    RandomForestClassifier(n_estimators=100, random_state=42))
])

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
pipe_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(n_estimators=100, random_state=42))
])

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
score_umap     = cross_val_score(pipe_umap,     digits.data, digits.target,
                                  cv=5, scoring="accuracy").mean()
score_baseline = cross_val_score(pipe_baseline, digits.data, digits.target,
                                  cv=5, scoring="accuracy").mean()

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
print(f"Tüm boyutlarla RF:         {score_baseline:.4f}")
print(f"UMAP(10D) + RF:            {score_umap:.4f}")

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
# ─── 5. PCA + t-SNE + UMAP Kapsamlı Karşılaştırma ───────────────
from sklearn.manifold import TSNE

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
cancer = load_breast_cancer()
X_c = StandardScaler().fit_transform(cancer.data)

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
methods = {
    "PCA (2D)"  : PCA(n_components=2, random_state=42),
    "t-SNE (2D)": TSNE(n_components=2, perplexity=30,
                        init="pca", random_state=42),
    "UMAP (2D)" : umap.UMAP(n_components=2, n_neighbors=15,
                             min_dist=0.1, random_state=42),
}

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, methods.items()):
    X_2d = model.fit_transform(X_c)
    ax.scatter(X_2d[:,0], X_2d[:,1],
               c=cancer.target, cmap="Set1", s=15, alpha=0.8)
    ax.set_title(f"{name}")
    ax.axis("off")

# --- Python: UMAP — Uygulama ve Pipeline Entegrasyonu ---
plt.suptitle("PCA vs t-SNE vs UMAP: Breast Cancer Görselleştirme")
plt.tight_layout(); plt.show()
